# ALSTM + L2 Tick因子（优化版v2）

## 优化点
1. Focal Loss 解决类别不平衡
2. 更强的模型架构（BatchNorm + Dropout）
3. 更多特征（原始OHLCV + L2 tick）
4. 学习率调度

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('请从 Quant 项目目录或其子目录启动 Jupyter')
sys.path.insert(0, str(PROJECT_ROOT))
from strategies.futures.alstm.data import load_minute_bars
output_root = PROJECT_ROOT / "artifacts" / "futures_alstm" / "alstm_l2tick_v2"
output_root.mkdir(parents=True, exist_ok=True)
print("完成")

In [ ]:
# 1) 加载L2 tick因子
frames = []
for month in range(7, 13):
    try:
        df = pd.read_parquet(PROJECT_ROOT / f"data_lake/future/market_data/tick_l2/year=2025/month={month:02d}.parquet")
        cffex = df[df['exchange'] == 'CFFEX']
        if_df = cffex[cffex['code'].str.startswith('IF')]
        if len(if_df) > 0: frames.append(if_df)
    except: pass
tick_df = pd.concat(frames).sort_values('timestamp').reset_index(drop=True)
tick_df['timestamp'] = pd.to_datetime(tick_df['timestamp'])
bid_vol = tick_df[['bid_vol_1','bid_vol_2','bid_vol_3','bid_vol_4','bid_vol_5']].sum(axis=1)
ask_vol = tick_df[['ask_vol_1','ask_vol_2','ask_vol_3','ask_vol_4','ask_vol_5']].sum(axis=1)
tick_df['l2_spread'] = (tick_df['ask_px_1'] - tick_df['bid_px_1']) / tick_df['last_price']
tick_df['l2_imbalance'] = (bid_vol - ask_vol) / (bid_vol + ask_vol + 1e-12)
tick_df['l2_mid_price'] = (tick_df['ask_px_1'] + tick_df['bid_px_1']) / 2
tick_df['l2_volume_change'] = tick_df['volume'].diff()
tick_df['l2_oi_change'] = tick_df['open_interest'].diff()
tick_df['l2_price_momentum'] = tick_df['last_price'].pct_change()
tick_df['minute'] = tick_df['timestamp'].dt.floor('1min')
l2_cols = ['l2_spread','l2_imbalance','l2_mid_price','l2_volume_change','l2_oi_change','l2_price_momentum']
tick_minute = tick_df.groupby('minute')[l2_cols].mean().reset_index().rename(columns={'minute':'datetime'})
tick_minute['datetime'] = pd.to_datetime(tick_minute['datetime'])
print(f"L2 tick: {len(tick_df):,} 行")

# 2) 加载分钟数据
raw_csv = load_minute_bars("IF")
raw_csv['datetime'] = pd.to_datetime(raw_csv['datetime'])

# 计算更多特征
raw_csv['returns_1min'] = raw_csv['vwap'].pct_change()
raw_csv['returns_5min'] = raw_csv['vwap'].pct_change(5)
raw_csv['returns_15min'] = raw_csv['vwap'].pct_change(15)
raw_csv['volatility_5'] = raw_csv['returns_1min'].rolling(5).std()
raw_csv['volatility_10'] = raw_csv['returns_1min'].rolling(10).std()
raw_csv['volatility_20'] = raw_csv['returns_1min'].rolling(20).std()
raw_csv['momentum_5'] = raw_csv['vwap'] / raw_csv['vwap'].shift(5) - 1
raw_csv['momentum_10'] = raw_csv['vwap'] / raw_csv['vwap'].shift(10) - 1
raw_csv['momentum_20'] = raw_csv['vwap'] / raw_csv['vwap'].shift(20) - 1
raw_csv['sma_5'] = raw_csv['vwap'].rolling(5).mean()
raw_csv['sma_10'] = raw_csv['vwap'].rolling(10).mean()
raw_csv['sma_20'] = raw_csv['vwap'].rolling(20).mean()
raw_csv['volume_ratio_5'] = raw_csv['volume'] / (raw_csv['volume'].rolling(5).mean() + 1e-12)
raw_csv['volume_ratio_10'] = raw_csv['volume'] / (raw_csv['volume'].rolling(10).mean() + 1e-12)
raw_csv['atr_5'] = (raw_csv['high'] / raw_csv['low'] - 1).rolling(5).mean()
raw_csv['atr_10'] = (raw_csv['high'] / raw_csv['low'] - 1).rolling(10).mean()
raw_csv['bb_position_20'] = (raw_csv['vwap'] - raw_csv['sma_20']) / (2 * raw_csv['vwap'].rolling(20).std() + 1e-12)
raw_csv['rsi_20'] = 100 * (raw_csv['returns_1min'].clip(lower=0).rolling(20).mean()) / (raw_csv['returns_1min'].abs().rolling(20).mean() + 1e-12)
raw_csv['close_open_ratio'] = raw_csv['close'] / (raw_csv['open'] + 1e-12)
raw_csv['high_low_ratio'] = raw_csv['high'] / (raw_csv['low'] + 1e-12)
raw_csv['price_position'] = (raw_csv['vwap'] - raw_csv['low']) / (raw_csv['high'] - raw_csv['low'] + 1e-12)

base_features = ['returns_1min','returns_5min','returns_15min','volatility_5','volatility_10','volatility_20',
    'momentum_5','momentum_10','momentum_20','sma_5','sma_10','sma_20','volume_ratio_5','volume_ratio_10',
    'atr_5','atr_10','bb_position_20','rsi_20','close_open_ratio','high_low_ratio','price_position']
all_features = base_features + l2_cols

# 合并
merged = raw_csv.merge(tick_minute, on='datetime', how='left')
merged[l2_cols] = merged[l2_cols].fillna(0)
merged = merged.dropna(subset=base_features)

# 标签
horizon, lag = 5, 1
future = merged['vwap'].shift(-(horizon + lag))
base = merged['vwap'].shift(-lag)
merged['future_return'] = future / base - 1
merged = merged.dropna(subset=['future_return'])

merged['label_trade'] = (merged['future_return'].abs() > 0.0005).astype(int)
merged['label_state'] = 1
merged.loc[merged['future_return'] > 0.0008, 'label_state'] = 2
merged.loc[merged['future_return'] < -0.0008, 'label_state'] = 0

train_mask = (merged['datetime'] >= '2024-07-01') & (merged['datetime'] <= '2025-06-30')
valid_mask = (merged['datetime'] >= '2025-07-01') & (merged['datetime'] <= '2025-09-30')
test_mask = (merged['datetime'] >= '2025-10-01') & (merged['datetime'] <= '2025-12-31')

print(f"特征: {len(all_features)} (分钟{len(base_features)} + L2 {len(l2_cols)})")
print(f"数据: {len(merged):,} 行")
print(f"Model A: {merged.loc[train_mask|valid_mask,'label_trade'].value_counts().to_dict()}")
print(f"Model B: {merged.loc[train_mask|valid_mask,'label_state'].value_counts().to_dict()}")

In [ ]:
# 3) 模型训练
step_len = 20

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        return focal_loss.mean()

class AttnLSTM(nn.Module):
    def __init__(self, nf, nc):
        super().__init__()
        self.bn = nn.BatchNorm1d(nf)
        self.lstm = nn.LSTM(nf, 64, 2, batch_first=True, dropout=0.3)
        self.attn = nn.Linear(64,1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, nc)
    def forward(self, x):
        # x: (batch, seq, features)
        bs, sl, nf = x.shape
        x = x.reshape(bs*sl, nf)
        x = self.bn(x)
        x = x.reshape(bs, sl, nf)
        o,_ = self.lstm(x)
        w = torch.softmax(self.attn(o),1)
        ctx = (o*w).sum(1)
        ctx = self.dropout(ctx)
        return self.fc(ctx)

def make_seq(X,y,sl):
    return np.array([X[i-sl:i] for i in range(sl,len(X))]), np.array([y[i] for i in range(sl,len(X))])

def train_model(mask, label, nc, name, cw=None):
    X = merged.loc[mask, all_features].values.astype(np.float32)
    y = merged.loc[mask, label].values.astype(np.int64)
    sc = StandardScaler(); X = sc.fit_transform(X)
    sp = int(len(X)*0.8)
    Xt,yt = make_seq(X[:sp],y[:sp],step_len)
    Xv,yv = make_seq(X[sp:],y[sp:],step_len)
    model = AttnLSTM(len(all_features),nc)
    opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    crit = FocalLoss(alpha=torch.FloatTensor(cw), gamma=2.0)
    best = float('inf'); pc = 0
    for ep in range(50):
        model.train()
        for xb,yb in DataLoader(TensorDataset(torch.FloatTensor(Xt),torch.LongTensor(yt)),128,shuffle=True):
            opt.zero_grad()
            crit(model(xb),yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = sum(crit(model(xb),yb).item() for xb,yb in DataLoader(TensorDataset(torch.FloatTensor(Xv),torch.LongTensor(yv)),128))/max(len(Xv),1)
        scheduler.step(vl)
        if vl<best: best=vl; torch.save(model.state_dict(),str(output_root/f'{name}.pt')); pc=0
        elif pc>=8: break
        else: pc+=1
    model.load_state_dict(torch.load(str(output_root/f'{name}.pt')))
    model.eval()
    Xe = merged.loc[test_mask,all_features].values.astype(np.float32)
    ye = merged.loc[test_mask,label].values.astype(np.int64)
    Xe = sc.transform(Xe)
    Xs,ys = make_seq(Xe,ye,step_len)
    with torch.no_grad(): pred = torch.argmax(model(torch.FloatTensor(Xs)),1).numpy()
    return (pred==ys).mean(), pred, ys

# 类别权重
cw_a = len(merged.loc[train_mask,'label_trade'])/(2*np.bincount(merged.loc[train_mask,'label_trade'].astype(int),minlength=2))
cw_b = len(merged.loc[train_mask,'label_state'])/(3*np.bincount(merged.loc[train_mask,'label_state'].astype(int),minlength=3))
print(f"权重A: {cw_a}")
print(f"权重B: {cw_b}")

acc_a,pred_a,true_a = train_model(train_mask|valid_mask,'label_trade',2,'model_a',cw_a)
print(f"\nModel A: {acc_a:.4f}")
acc_b,pred_b,true_b = train_model(train_mask|valid_mask,'label_state',3,'model_b',cw_b)
print(f"Model B: {acc_b:.4f}")

In [ ]:
print('='*60)
print('结果')
print('='*60)
print(f'特征: {len(all_features)} (分钟{len(base_features)} + L2 {len(l2_cols)})')
print(f'\nModel A: {acc_a:.4f}')
labels_a = ['no_trade','trade']
for c in range(2):
    m=true_a==c
    if m.any():
        name = labels_a[c]
        acc = (pred_a[m]==c).mean()
        cnt = m.sum()
        print(f'  {name}: {acc:.4f} ({cnt})')
print(f'\nModel B: {acc_b:.4f}')
labels_b = ['down','flat','up']
for c in range(3):
    m=true_b==c
    if m.any():
        name = labels_b[c]
        acc = (pred_b[m]==c).mean()
        cnt = m.sum()
        print(f'  {name}: {acc:.4f} ({cnt})')
json.dump({'features':all_features,'model_a':float(acc_a),'model_b':float(acc_b)},open(output_root/'results.json','w'),indent=2)
print(f'\n结果已保存')